# Comparador Automatizado de Normativas vs Manuales Internos

**Pipeline de 5 fases** para analizar el cumplimiento de manuales internos bancarios
respecto a normativas ecuatorianas (SBS, BCE, SEPS, UAF).

| Fase | Descripción | Módulo |
|------|-------------|--------|
| 1 | Tabulación de documentos (PDF → DataFrame) | `document_parser` |
| 2 | Motor de búsqueda (FAISS semántico + léxico + reranker) | `search_engine` |
| 3 | Retrieve-then-Grade (validación de candidatos via LLM) | `llm_grader` |
| 4 | Análisis comparativo + NER (LLM) | `llm_grader` |
| 5 | Procesamiento concurrente (ThreadPoolExecutor + tqdm) | `comparator` |

**Modelos Docker Model Runner disponibles:**
- 🔵 Embedding: `ai/qwen3-embedding:latest` (2560 dim) — mejor calidad semántica en español
- 🔵 Embedding rápido: `ai/granite-embedding-multilingual:latest` (768 dim)
- 🟢 LLM: `docker.io/ai/gemma4:latest` — razonamiento interno (CoT), ideal para análisis legal
- 🟡 Reranker: `docker.io/ai/qwen3-reranker-vllm:0.6B` — filtrado post-FAISS

## ⚙️ 0. Configuración

In [ ]:
import os
import sys
import logging
from pathlib import Path

# Apple Silicon 16GB: evita el crash nativo del kernel cuando faiss y el
# runtime OpenMP de Docling/torch coexisten en el mismo proceso (Fase 2.2)
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

# Agregar src/ al path
sys.path.insert(0, str(Path.cwd()))

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    datefmt='%H:%M:%S',
)

# Directorios de entrada y salida
NORMATIVA_DIR = Path("Normativa2026")          # PDFs de normativas
MANUAL_DIR    = Path("document_test")           # PDFs de manuales internos
OUTPUT_DIR    = Path("output/comparador")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Normativas: {list(NORMATIVA_DIR.glob('*.pdf'))}")
print(f"Manuales:   {list(MANUAL_DIR.glob('*.pdf'))}")

In [ ]:
# Importar todos los módulos del comparador
from src import (
    NormativaParser,
    ManualParser,
    LangChainDMREmbeddings,
    SentenceTransformersEmbeddings,
    NormativaIndex,
    LLMGrader,
    DocumentComparator,
)
from src.config import (
    DMR_BASE_URL,
    DMR_EMBED_MODEL,
    DMR_LLM_MODEL,
    FAISS_TOP_K,
    RERANKER_TOP_N,
)

print("✅ Módulos cargados correctamente")
print(f"   Embedding model : {DMR_EMBED_MODEL}")
print(f"   LLM model       : {DMR_LLM_MODEL}")
print(f"   DMR base URL    : {DMR_BASE_URL}")

## 📄 Fase 1: Tabulación de Documentos

### 1.1 Normativas (Docling + regex)

In [ ]:
from tqdm.notebook import tqdm
import pandas as pd

# cache_dir apunta a los markdowns ya generados por Docling (no requiere reconversión)
normativa_parser = NormativaParser(cache_dir="output/docling")

# Stems explícitos para evitar intentar convertir L1-XVI-cap-*.pdf (no cacheados → segfault MPS)
NORMATIVA_STEMS = [
    "PDL-DERECHOS-DIGITALES",
    "LEY-ORGANICA-PARA-EL-FORTALECIMIENTO-DE-LA-CIBERSEGURIDAD_202652616421988",
    "Proyecto-de-Ley-Organica-Organica-para-Reprimir-y-Prevenir-el-Lavado-de-Activos-y-la-Financiacion-del-Terrorismo",
    "Proyecto-de-Ley-Transformacion-Digital-y-Audiovisual",
    "Resoluci_n_N_SPDP_SPD_2026_0009_R_1771536870",
]
normativa_pdfs = [NORMATIVA_DIR / f"{s}.pdf" for s in NORMATIVA_STEMS]
normativa_frames = []

for pdf in tqdm(normativa_pdfs, desc="Parseando normativas"):
    df = normativa_parser.parse_pdf(pdf)
    normativa_frames.append(df)
    print(f"  {pdf.name}: {len(df)} elementos")

normativa_df = pd.concat(normativa_frames, ignore_index=True)
print(f"\nTotal: {len(normativa_df)} elementos normativos")
normativa_df.head(3)

In [ ]:
normativa_df.to_excel("output/docling/normativa_df.xlsx", index=False)

In [ ]:
# Resumen de la normativa
print("Distribución por tipo de elemento:")
display(normativa_df["tipo_elemento"].value_counts().to_frame("cantidad"))

print("\nDistribución por documento:")
display(normativa_df["doc_id"].value_counts().to_frame("elementos"))

# Guardar normativa procesada
normativa_df.to_json(
    OUTPUT_DIR / "normativa_tabulada.json",
    orient="records",
    force_ascii=False,
    indent=2,
)
print("\n✅ Normativa guardada en output/comparador/normativa_tabulada.json")

In [ ]:
normativa_df.query('doc_id=="PDL-DERECHOS-DIGITALES.pdf"').sample(1).values

### 1.2 Manuales internos (Docling + HybridChunker)

> **Requiere `document_test/` con los manuales.** No viajan en git por confidencialidad.
> Si no los tienes, salta a las **Fases 6 y 7**: verifican las mejoras del pipeline con un
> corpus sintético, sin necesidad de documentos del cliente.

In [ ]:
manual_parser = ManualParser(device="cpu")  # MPS inestable en conversiones sin caché → CPU estable

manual_pdfs = sorted(MANUAL_DIR.glob("*.pdf"))

# Los manuales internos NO viajan en git: document_test/ está en .gitignore por
# confidencialidad (ver README → Confidencialidad). Sin este aviso, la celda moría más
# abajo con "ValueError: No objects to concatenate" de pandas — un error que no dice
# nada del problema real y manda a depurar al sitio equivocado.
if not manual_pdfs:
    raise FileNotFoundError(
        f"No hay ningún PDF en {MANUAL_DIR}/.\n\n"
        "Los manuales internos no se versionan por confidencialidad. Cópialos a ese "
        "directorio antes de ejecutar esta celda.\n\n"
        "Si solo quieres verificar las mejoras del pipeline, las Fases 6 y 7 de este "
        "notebook corren con un corpus sintético y no necesitan estos documentos."
    )

manual_frames = []
for pdf in tqdm(manual_pdfs, desc="Parseando manuales"):
    df = manual_parser.parse_pdf(pdf)
    manual_frames.append(df)
    print(f"  {pdf.name}: {len(df)} chunks")

manual_df = pd.concat(manual_frames, ignore_index=True)
print(f"\nTotal: {len(manual_df)} secciones del manual")
manual_df.head(3)

In [ ]:
# Vista previa del manual
print(f"Secciones totales: {len(manual_df)}")
print(f"Documentos: {manual_df['doc_id'].unique()}")
print("\nPrimeros chunks:")
display(manual_df[["chunk_id", "jerarquia", "titulo_seccion", "texto"]].head(5))

manual_df.to_json(
    OUTPUT_DIR / "manual_tabulado.json",
    orient="records",
    force_ascii=False,
    indent=2,
)
print("✅ Manual guardado en output/comparador/manual_tabulado.json")

## 🔍 Fase 2: Motor de Búsqueda

### 2.1 Inicializar backend de embeddings

In [ ]:
# ai/qwen3-embedding (2560 dim) — preferido, pero requiere GPU funcional
# ai/granite-embedding-multilingual (768 dim) — alternativa estable
embedding_backend = LangChainDMREmbeddings(
    model="ai/granite-embedding-multilingual:latest",   # 768 dim, estable
    # model="ai/qwen3-embedding:latest",               # 2560 dim, mayor calidad
    base_url=DMR_BASE_URL,
)

import numpy as np
test_vec = embedding_backend.encode(["prueba de conexión DMR"])
print(f"✅ Embedding backend OK")
print(f"   Modelo    : {embedding_backend.model_name}")
print(f"   Dimensión : {test_vec.shape[1]}")
print(f"   Norma L2 (debe ≈ 1.0): {np.linalg.norm(test_vec[0]):.4f}")

### 2.2 Construir índice FAISS sobre la normativa

In [ ]:
from tqdm.notebook import tqdm as tqdmn

normativa_index = NormativaIndex(
    embedding_backend=embedding_backend,
    use_reranker=True,  # CrossEncoder local (sentence-transformers) — vllm-metal no soporta reranking
)

# Solo indexar artículos (excluir disposiciones/anexos si se prefiere)
# normativa_arts = normativa_df[normativa_df["tipo_elemento"] == "articulo"]

print("Construyendo índice FAISS...")
normativa_index.build(normativa_df, text_col="embed_text")

# Persistir en disco
normativa_index.save(OUTPUT_DIR / "faiss_index")
print(f"✅ Índice FAISS guardado en output/comparador/faiss_index/")

### 2.3 Verificar búsquedas

In [ ]:
# Test: búsqueda semántica
query = "política de crédito y evaluación de riesgo crediticio"
resultados = normativa_index.semantic_search(query, top_k=5)

print(f"Búsqueda: '{query}'")
print(f"Top-{len(resultados)} resultados FAISS:")
for r in resultados:
    print(f"  [{r['rank_faiss']}] Art.{r.get('numero','?')} (sim={r['similarity']:.4f}): {r.get('encabezado','')[:60]}")

In [ ]:
# Test: reranking post-FAISS
reranked = normativa_index.rerank(query, resultados, top_n=3)

print(f"Top-{len(reranked)} tras reranking:")
for r in reranked:
    score_str = f"reranker={r.get('reranker_score', 'N/A')}" if 'reranker_score' in r else f"sim={r.get('similarity', 0):.4f}"
    print(f"  [{r.get('rank', '?')}] Art.{r.get('numero','?')} ({score_str}): {r.get('encabezado','')[:60]}")

In [ ]:
# Test: búsqueda léxica
texto_manual = "Según el artículo 5 y el artículo 12 de la normativa vigente, el banco debe..."
lexical = normativa_index.lexical_scan(texto_manual, normativa_df)

print(f"Referencias léxicas detectadas: {len(lexical)}")
for m in lexical:
    print(f"  Art.{m.get('numero','?')}: {m.get('encabezado','')[:70]}")

## 🤖 Fase 3+4: LLM Grader — Retrieve-then-Grade + Análisis Comparativo

In [ ]:
llm_grader = LLMGrader(
    model=DMR_LLM_MODEL,   # docker.io/ai/gemma4:latest
    base_url=DMR_BASE_URL,
    temperature=0.0,
)

print(f"✅ LLM Grader inicializado")
print(f"   Modelo: {DMR_LLM_MODEL}")
print(f"   Nota: gemma4 tiene razonamiento interno (CoT) → latencia mayor, mayor calidad")

In [ ]:
# Test grading en una sección del manual
sample_row = manual_df.iloc[0].to_dict()
sample_text = sample_row.get("texto", "")

print(f"Sección del manual: {sample_row.get('jerarquia','')[:80]}")
print(f"Texto (primeros 300 chars): {sample_text[:300]}\n")

# Búsqueda de candidatos
candidates = normativa_index.semantic_search(sample_text, top_k=5)
candidates = normativa_index.rerank(sample_text, candidates, top_n=3)

print(f"Candidatos FAISS recuperados: {len(candidates)}")
for c in candidates:
    print(f"  Art.{c.get('numero','?')}: {c.get('encabezado','')[:60]}")

In [ ]:
# Grading de candidatos
print("Evaluando relevancia de candidatos...")
graded = llm_grader.grade_candidates(sample_text, candidates)

print("\nResultados del grading:")
for g in graded:
    status = "✅" if g.get("relevante") else "❌"
    print(f"  {status} Art.{g.get('numero','?')} | score={g.get('score_grade',0):.2f} | {g.get('razon_grade','')[:70]}")

In [ ]:
# Análisis comparativo completo
lexical_matches = normativa_index.lexical_scan(sample_text, normativa_df)
validated = [c for c in graded if c.get("relevante", True)]

print("Ejecutando análisis comparativo...")
analysis = llm_grader.analyze_comparison(sample_row, lexical_matches, validated)

print(f"\n{'='*60}")
print(f"Tipo de coincidencia : {analysis.tipo_coincidencia}")
print(f"Nivel de cumplimiento: {analysis.nivel_cumplimiento}")
print(f"\nAnálisis general:")
print(analysis.analisis_general)
if analysis.brechas:
    print(f"\nBrechas detectadas ({len(analysis.brechas)}):")
    for b in analysis.brechas:
        print(f"  • {b}")
if analysis.entidades_normativas:
    print(f"\nEntidades normativas: {analysis.entidades_normativas}")
if analysis.entidades_financieras:
    print(f"Entidades financieras: {analysis.entidades_financieras}")

## ⚡ Fase 5: Pipeline Completo con Procesamiento Concurrente

In [ ]:
comparator = DocumentComparator(
    normativa_index=normativa_index,
    llm_grader=llm_grader,
    top_k_faiss=FAISS_TOP_K,
    top_n_rerank=RERANKER_TOP_N,
)

print("DocumentComparator inicializado")
print(f"  FAISS top-k     : {FAISS_TOP_K}")
print(f"  Reranker top-n  : {RERANKER_TOP_N}")

In [ ]:
# Muestra rápida de 3 secciones para validar el pipeline antes del run completo
print("Ejecutando pipeline en muestra de 3 secciones...")
sample_results = comparator.run_sample(
    manual_df=manual_df,
    normativa_df=normativa_df,
    n=3,
    max_workers=1,
)

print(f"\n✅ Muestra completada: {len(sample_results)} secciones analizadas")
display(sample_results[["chunk_id", "jerarquia", "tipo_coincidencia", "nivel_cumplimiento", "analisis_general"]].head())

In [ ]:
# Pipeline sobre muestra de validación
# M1 16GB: gemma4 CoT secuencial en GPU → ~2-4 min/chunk (grading + análisis)
# n=5 → ~20-40 min total; aumentar a n=20 solo si se confirma estabilidad
print("Ejecutando pipeline sobre muestra de validación (n=5, max_workers=1)...")
results_df = comparator.run_sample(
    manual_df=manual_df,
    normativa_df=normativa_df,
    n=5,
    max_workers=1,
)

print(f"\n✅ Pipeline completado: {len(results_df)} secciones analizadas")
display(results_df[["chunk_id", "jerarquia", "tipo_coincidencia", "nivel_cumplimiento"]].head(5))

## 📊 Resultados y Exportación

In [ ]:
# Resumen estadístico
print("=== RESUMEN DE CUMPLIMIENTO ===")
summary = DocumentComparator.summary(results_df)
display(summary)

print("\n=== DISTRIBUCIÓN POR TIPO DE COINCIDENCIA ===")
display(results_df["tipo_coincidencia"].value_counts().to_frame("secciones"))

In [ ]:
# Secciones con omisiones críticas
omisiones = results_df[
    results_df["nivel_cumplimiento"].isin(["omision", "parcial"])
].copy()

print(f"Secciones con omisiones/cumplimiento parcial: {len(omisiones)}")
display(omisiones[["jerarquia", "tipo_coincidencia", "nivel_cumplimiento", "analisis_general"]].head(10))

In [ ]:
# Exportar a Excel formateado
excel_path = comparator.export_excel(
    results_df,
    output_path=OUTPUT_DIR / "reporte_comparacion.xlsx",
)
print(f"✅ Reporte exportado: {excel_path}")

# También guardar JSON completo
results_df.to_json(
    OUTPUT_DIR / "reporte_comparacion.json",
    orient="records",
    force_ascii=False,
    indent=2,
)
print(f"✅ JSON exportado: {OUTPUT_DIR / 'reporte_comparacion.json'}")

## 🔄 Uso Modular (sin re-ejecutar todo el pipeline)

Para análisis incrementales, carga el índice FAISS desde disco:

In [ ]:
# Cargar índice existente sin re-indexar
# IMPORTANTE: usar granite (768d) para coincidir con el índice FAISS guardado
from src import LangChainDMREmbeddings, NormativaIndex, LLMGrader, DocumentComparator
from src.config import DMR_BASE_URL, DMR_LLM_MODEL

embed_backend = LangChainDMREmbeddings(
    model="ai/granite-embedding-multilingual:latest",   # 768d — coincide con índice guardado
    base_url=DMR_BASE_URL,
)

saved_index = NormativaIndex(embed_backend)
saved_index.load(OUTPUT_DIR / "faiss_index")

grader = LLMGrader(model=DMR_LLM_MODEL, base_url=DMR_BASE_URL)
comp = DocumentComparator(saved_index, grader)

print("✅ Pipeline cargado desde disco")
print(f"   Vectores en índice: {saved_index._index.ntotal}")
print(f"   Dimensión embedding: {saved_index._index.d}")

In [ ]:
# Análisis incremental: validar recarga sin re-indexar
# Usar run_sample(n=3) para validación rápida (max_workers=1 para no sobrecargar DMR)
import pandas as pd

normativa_df = pd.read_json(OUTPUT_DIR / "normativa_tabulada.json", orient="records")
new_manual_df = pd.read_json(OUTPUT_DIR / "manual_tabulado.json", orient="records")

incremental = comp.run_sample(
    manual_df=new_manual_df,
    normativa_df=normativa_df,
    n=3,
    max_workers=1,
)

display(incremental[["chunk_id", "nivel_cumplimiento", "analisis_general"]].head())

## Comparativa pruebas

In [ ]:
incremental.head()

## ✅ Fase 6: Verificación de subsanaciones

Cada celda demuestra **un defecto corregido** en las olas 0-2, contrastando el
comportamiento actual con el que tenía antes.

Corre en segundos y **no necesita `document_test/` ni un modelo vivo**: usa el corpus
sintético de `tests/fixtures/`, que construye a propósito los casos límite (artículo
huérfano, sección que cubre dos normas, mismo número de artículo en dos normativas).

> Las secciones anteriores demuestran que el pipeline *funciona*. Esta demuestra que
> **no miente cuando algo falla**, que es un requisito distinto y el que motivó el plan.


In [ ]:
from tests.fixtures import FakeChatModel, FakeGrader, FakeIndex, manual_df, normativa_df

normativa_sintetica = normativa_df()   # LEY-A (6 arts.) + RES-B (4 arts.)
manual_sintetico = manual_df()         # 8 secciones

print(f"Corpus de verificación: {len(normativa_sintetica)} artículos · "
      f"{len(manual_sintetico)} secciones")
print("Material inventado — no depende de documentos del cliente.")

### 6.1 · Ítem 1 — el modelo cae a mitad de la corrida

**Antes:** toda excepción producía `nivel_cumplimiento="no_aplica"` con el error dentro
del análisis. El papel de trabajo afirmaba "no aplica" sobre secciones que el modelo
nunca llegó a leer — un fallo disfrazado de resultado.

**Ahora:** la corrida se detiene, dice cuánto quedó sin procesar, y `estado_analisis`
distingue el fallo técnico del veredicto.

In [ ]:
import openai

from src.comparator import DocumentComparator
from src.errors import RunAbortedError

comparador_caido = DocumentComparator(
    normativa_index=FakeIndex(normativa_df=normativa_sintetica, plan={}),
    llm_grader=FakeGrader(fallar_en=lambda fila: openai.APIConnectionError(request=None)),
)

try:
    comparador_caido.run(manual_sintetico, normativa_sintetica, max_workers=1)
    print("✗ la corrida NO se detuvo")
except RunAbortedError as e:
    veredictos = set(e.parciales["nivel_cumplimiento"].dropna())
    print(f"✓ abortó tras {e.completadas}/{e.total} · {e.omitidas} sin procesar")
    print(f"✓ veredictos 'no_aplica' emitidos: {len(veredictos & {'no_aplica'})}   (antes: todas las filas)")
    print(f"✓ estado por fila: {dict(e.parciales['estado_analisis'].value_counts())}")
    print("\n  'omitido' = nadie las miró.  'error_modelo' = esta disparó el aborto.")
    print("  Ninguna finge ser un veredicto de cumplimiento.")

### 6.2 · Ítem 4 — el grading devuelve algo que no parsea

**Antes:** `logger.warning("Se asumen todos relevantes")` y `relevante=True` para todos.
Falsos positivos de cumplimiento colados en silencio — el riesgo espejo del 6.1, y peor:
allí se perdían secciones de forma ruidosa, aquí se afirmaba haber verificado algo que
nadie verificó.

In [ ]:
from src.llm_grader import LLMGrader

grader_roto = LLMGrader(
    chat_grader=FakeChatModel(respuestas=["esto no es JSON"] * 6),
    chat_analyst=FakeChatModel(),
)
candidatos = [{"element_id": "a1", "contenido": "x"}, {"element_id": "a2", "contenido": "y"}]
salida = grader_roto.grade_candidates("sección del manual", candidatos)

print(f"relevante=True : {sum(c['relevante'] is True for c in salida)}   (antes: {len(candidatos)})")
print(f"relevante=None : {sum(c['relevante'] is None for c in salida)}   → van a revisión manual")
print(f"motivo         : {salida[0]['motivo_revision']}")
print("\nTres intentos antes de rendirse: cadena normal → reparar JSON → prompt mínimo.")

### 6.3 · §2.2 — el mismo número de artículo en dos normativas

El manual cita *"Conforme al Art. 5"* sin decir de cuál norma. Ese número existe en las
dos cargadas.

**Antes:** matcheaba las dos como citas firmes, generando una arista de cobertura falsa
hacia la norma que el manual no citó.

**Ahora:** se emiten etiquetadas. Ni ambas como ciertas, ni una elegida a dedo, ni
descartadas — descartar borraría el hecho de que el manual *sí* cita un artículo, y ese
dato lo necesitan el modelo N:N y el flag de revisión manual.

In [ ]:
indice = FakeIndex(normativa_df=normativa_sintetica, plan={})
seccion = manual_sintetico[manual_sintetico["jerarquia"] == "2.2 Registro y conservación"].iloc[0]

for m in indice.lexical_scan(seccion["texto"], normativa_df=normativa_sintetica):
    print(f"  {m['doc_id']:22} Art.{m['numero']:3}  tipo={m['match_type']:8} score={m['similarity']}")
    print(f"     └─ {m['razon_match']}")

print("\nUna cita ambigua NO cuenta como coincidencia léxica: el análisis la recibe")
print("declarada como indicio, no como referencia confirmada.")

### 6.4 · §2.2 — el umbral de score semántico

**Antes:** `_process_row()` llamaba a `semantic_search()` sin `min_score`, así que el
slider de la barra lateral era decorativo: la corrida siempre usaba el default.

In [ ]:
indice_espia = FakeIndex(normativa_df=normativa_sintetica, plan={})
DocumentComparator(
    normativa_index=indice_espia, llm_grader=FakeGrader(), min_semantic_score=0.77,
).run(manual_sintetico.head(1), normativa_sintetica, max_workers=1)

print(f"min_score que recibió el índice: {indice_espia.min_score_recibido}   (antes: siempre 0.30)")

### 6.5 · §3.3.2 y S10 — trazabilidad y separación de capas

**Antes:** todas las corridas escribían a `output/comparador/reporte_comparacion.xlsx`.
Dos corridas se pisaban el reporte sin forma de saber cuál produjo cuál — inaceptable en
un papel de trabajo, donde la trazabilidad es el punto.

Y la orquestación vivía dentro de la UI, de modo que ninguna otra interfaz podía
reutilizarla.

In [ ]:
from src.service import RunPaths, nuevo_run_id

a, b = RunPaths(run_id=nuevo_run_id()), RunPaths(run_id=nuevo_run_id())
print(f"corrida A → {a.excel}")
print(f"corrida B → {b.excel}")
print(f"¿colisionan? {a.excel == b.excel}   (antes: siempre la misma ruta)\n")

fuente_ui = open("streamlit_app.py").read()
constructores = [c for c in ("NormativaParser(", "ManualParser(", "NormativaIndex(",
                             "LLMGrader(", "DocumentComparator(") if c in fuente_ui]
print(f"construcciones del pipeline en la UI: {len(constructores)}   (antes: 5)")
print(f"llamadas a src/service.py:            {fuente_ui.count('service.')}")
print("\nLa UI es una vista. FastAPI, en Fase 2, será otra sobre el mismo servicio.")

### 6.6 · Ítem T — una sola fuente de color

**Antes:** la paleta estaba escrita a mano en dos archivos (`app/theme.py` y los
`PatternFill` del Excel). "Coincidían" porque alguien los copió.

In [ ]:
from app.theme import NIVEL_COLORS
from src.design_tokens import PRIMARIO, marca, ratio_contraste, tinte

print(f"tinte('cumple')  tokens={tinte('cumple')}  tema={NIVEL_COLORS['cumple']['bg']}  "
      f"¿iguales? {tinte('cumple') == NIVEL_COLORS['cumple']['bg']}\n")

print("Trazo de gráfica vs. relleno de tabla — no pueden ser el mismo color:")
for nivel in ("cumple", "parcial", "omision", "no_aplica"):
    print(f"  {nivel:10} marca={marca(nivel)} ({ratio_contraste(marca(nivel), '#ffffff'):5.2f}:1)  "
          f"tinte={tinte(nivel)}")
print(f"\nUn tinte da ~1.2:1 sobre blanco: como barra de gráfica sería invisible.")
print(f"Y ninguna marca puede ser el verde institucional ({PRIMARIO}) o se confundiría")
print("el dato con el cromo de la interfaz.")

---

### Qué NO cubre esta sección

- **El pipeline con documentos reales.** Requiere `document_test/`, que está fuera de git
  por confidencialidad. Las secciones 1-5 de este notebook son las que lo ejercitan.
- **Los ítems 2 y 3** (sincronía de estado al refrescar, checkpoints): pendientes en
  `feature/run-manager`.
- **Los ítems 5-10** (modelo N:N, doble vía, alcance, revisión manual, chunking, papel de
  trabajo): olas 3 y 4.

Estado completo en `PLAN_MEJORAS_ANEXO.md`.

## 🔀 Fase 7: Doble vía y modelo de cobertura (Ola 3)

La Fase 6 demuestra que el pipeline **no miente cuando algo falla**. Esta demuestra que
ahora **puede ver lo que antes no veía**.

El cambio de fondo: hasta la Ola 2, el resultado era una fila por sección con los
artículos aplanados a texto. Eso responde *"¿cumple esta sección?"* pero no
*"¿queda algún artículo sin cubrir?"* — y esa segunda pregunta es la premisa no
negociable del Bloque A.

In [ ]:
from src.comparator import DocumentComparator
from src.coverage import ORIGEN_LEXICO, ORIGEN_SEMANTICO_V1, CoverageLink, LinkTable
from src.dual import run_dual
from tests.fixtures import FakeGraderDual, FakeIndex, FakeManualIndex

normativa_sintetica, manual_sintetico = normativa_df(), manual_df()
print(f"{len(normativa_sintetica)} artículos · {len(manual_sintetico)} secciones")

### 7.1 · Ítem 5 — la consulta que antes exigía re-correr todo

**Antes:** `articulos_lexicos` era una lista de números dentro de la fila de cada sección.
Para saber qué secciones cubren el Art. 35 había que recorrer todas las filas y parsear
texto — o volver a ejecutar el pipeline entero.

**Ahora:** la arista es la unidad, y se consulta en ambos sentidos.

In [ ]:
tabla = LinkTable([
    CoverageLink(articulo_element_id="LEY_35", articulo_numero="35",
                 seccion_chunk_id="S1", seccion_jerarquia="4.1 Conocimiento del cliente",
                 relevante=True, score_semantico=0.82, origen=ORIGEN_LEXICO),
    CoverageLink(articulo_element_id="LEY_35", articulo_numero="35",
                 seccion_chunk_id="S7", seccion_jerarquia="8.1 Expedientes",
                 relevante=True, score_semantico=0.61),
])

print("¿Qué secciones cubren el Art. 35?")
for e in tabla.por_articulo("LEY_35"):
    print(f"  {e.seccion_jerarquia:32} score={e.confianza}  origen={e.origen}")

print("\n¿Qué artículos toca la sección S1?")
for e in tabla.por_seccion("S1"):
    print(f"  Art. {e.articulo_numero}")

### 7.2 · Ítem 5 — el resultado no depende del orden

Las dos vías recorren el mismo grafo en sentidos opuestos y llegan a las mismas parejas.
Si al fusionarlas ganara *la última observación*, el papel de trabajo cambiaría según en
qué orden corrieron las vías — un no-determinismo inaceptable en auditoría.

`upsert` conserva **lo mejor de cada observación**, no la última.

In [ ]:
def _par(origen, score):
    return CoverageLink(articulo_element_id="A", seccion_chunk_id="S",
                        origen=origen, score_semantico=score)

a = LinkTable([_par(ORIGEN_SEMANTICO_V1, 0.6), _par(ORIGEN_LEXICO, 0.9)])
b = LinkTable([_par(ORIGEN_LEXICO, 0.9), _par(ORIGEN_SEMANTICO_V1, 0.6)])
ea, eb = next(iter(a)), next(iter(b))

print(f"orden 1 → origen={ea.origen:16} score={ea.score_semantico}")
print(f"orden 2 → origen={eb.origen:16} score={eb.score_semantico}")
print(f"¿idénticos? {(ea.origen, ea.score_semantico) == (eb.origen, eb.score_semantico)}")
print(f"orígenes acumulados: {ea.origenes}   ← que ambas vías coincidan es evidencia más fuerte")

### 7.3 · Ítem 6 — la alerta que la Vía 1 sola **no puede producir**

Es el argumento entero del ítem: si ninguna sección recupera un artículo, ese artículo
**no aparece** en la Vía 1. Un artículo que nadie analizó es indistinguible de uno que no
existe.

La brecha solo se ve recorriendo desde el otro lado.

In [ ]:
grader = FakeGraderDual()
comparador = DocumentComparator(
    normativa_index=FakeIndex(normativa_df=normativa_sintetica, plan={}),
    llm_grader=grader,
)
bundle = run_dual(
    comparador=comparador,
    manual_index=FakeManualIndex(manual_sintetico),
    manual_df=manual_sintetico,
    normativa_df=normativa_sintetica,
)

print(f"Vía 1 → {len(bundle.vista_manual)} secciones")
print(f"Vía 2 → {len(bundle.vista_normativa)} artículos sustantivos "
      f"(las referencias del preámbulo se excluyen: no son obligaciones)\n")
print(f"Cobertura: {bundle.cobertura.porcentaje:.0%}")
print(f"⚠ {bundle.alerta_cobertura}\n")
print("Artículos sin cubrir:")
for a in bundle.cobertura.sin_cobertura[:4]:
    print(f"  {a['doc_id']:22} Art.{a['numero']:4} {a['encabezado'][:42]}")

### 7.4 · Ítem 6 — el coste es la **suma**, no el producto

Las dos vías evalúan las mismas parejas (artículo, sección) por caminos distintos. Sin
caché compartida, el coste sería el producto — la diferencia entre una corrida y dos
(supuesto S3 del plan).

Es el DoD del ítem, y aquí se mide en vez de afirmarse.

In [ ]:
n_sec = len(manual_sintetico)
n_art = bundle.cobertura.total_articulos
total = grader.llamadas_grade + grader.llamadas_adopcion

print(f"llamadas reales : {grader.llamadas_grade} (Vía 1) + {grader.llamadas_adopcion} (Vía 2) = {total}")
print(f"suma esperada   : {n_sec} + {n_art} = {n_sec + n_art}")
print(f"sin caché sería : {n_sec} × {n_art} = {n_sec * n_art}")
print(f"\n¿cumple el DoD? {total == n_sec + n_art}")

### 7.5 · Ítem 6 — cobertura completa: la alerta desaparece

Si toda la normativa queda cubierta, no hay banner. La alerta no es decorativa: aparece
exactamente cuando la premisa del Bloque A no se cumple.

In [ ]:
arts = normativa_sintetica[
    (normativa_sintetica["tipo_elemento"] == "articulo")
    & (~normativa_sintetica["es_referencia"])
]
plan_completo = {r["embed_text"]: [manual_sintetico.iloc[0]["chunk_id"]]
                 for _, r in arts.iterrows()}

completo = run_dual(
    comparador=DocumentComparator(
        normativa_index=FakeIndex(normativa_df=normativa_sintetica, plan={}),
        llm_grader=FakeGraderDual(),
    ),
    manual_index=FakeManualIndex(manual_sintetico, plan_completo),
    manual_df=manual_sintetico,
    normativa_df=normativa_sintetica,
)
print(f"cobertura: {completo.cobertura.porcentaje:.0%}")
print(f"alerta:    {completo.alerta_cobertura}   ← None significa que no hay brecha")

---

### Qué falta de la Ola 3

- **UI de la doble vía** — sub-pestañas por vía y tarjeta de cobertura, con el design
  system (`marca()` para gráficas, `tinte()` para tablas, icono y etiqueta obligatorios).
- **Ítem 7** — selector de alcance por lista. §12.1 quedó confirmado: **dos selectores**,
  uno por vía.
- **Ítem 10** — flag de revisión manual. El modelo ya lo transporta
  (`requiere_revision`, `motivos_revision`); falta la vista y la hoja del Excel.

Estado completo en `PLAN_MEJORAS_ANEXO.md`.